<a href="https://colab.research.google.com/github/muneer-ahmad10/Resume-Analyzer/blob/main/Resume_Analyzer_streamli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pyngrok PyPDF2 sentence-transformers scikit-learn matplotlib spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 55.6 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import re
import PyPDF2
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ==================================================
# PAGE TITLE
# ==================================================

st.title("🚀 AI Resume Analyzer")

st.write("Upload Resume + Paste Job Description")

# ==================================================
# LOAD MODEL
# ==================================================

model = SentenceTransformer('all-MiniLM-L6-v2')

# ==================================================
# CLEAN TEXT
# ==================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9 ]', ' ', text)

    text = " ".join(text.split())

    return text

# ==================================================
# EXTRACT PDF TEXT
# ==================================================

def extract_text_from_pdf(file):

    reader = PyPDF2.PdfReader(file)

    text = ""

    for page in reader.pages:

        extracted = page.extract_text()

        if extracted:
            text += extracted

    return text

# ==================================================
# SKILLS DATABASE
# ==================================================

SKILLS_DB = [

    "python",
    "machine learning",
    "deep learning",
    "nlp",
    "sql",
    "pytorch",
    "tensorflow",
    "streamlit",
    "rag",
    "azure",
    "mlops",
    "api",
    "apis",
    "vector databases",
    "semantic search",
    "document understanding",
    "llms",
    "data analysis",
    "pandas",
    "numpy",
    "scikit learn",
    "xgboost",
    "transformers"

]

# ==================================================
# SKILL EXTRACTION
# ==================================================

def extract_skills(text):

    text = text.lower()

    found_skills = []

    for skill in SKILLS_DB:

        if skill in text:
            found_skills.append(skill)

    return list(set(found_skills))

# ==================================================
# SIMILARITY
# ==================================================

def compute_similarity(resume, job):

    emb1 = model.encode([resume])

    emb2 = model.encode([job])

    score = cosine_similarity(emb1, emb2)[0][0]

    return round(score * 100, 2)

def extract_email(text):

    pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]+'

    emails = re.findall(pattern, text)

    if emails:
        return emails[0]

    return "Not Found"

def extract_phone(text):

    pattern = r'(\+?\d[\d\s\-]{8,15}\d)'

    phones = re.findall(pattern, text)

    if phones:
        return phones[0]

    return "Not Found"

def extract_name(text):

    lines = text.split("\n")

    for line in lines:

        line = line.strip()

        if len(line.split()) >= 2:

            if line.isupper():

                return line.title()

    return "Not Found"



# ==================================================
# USER INPUTS
# ==================================================

uploaded_file = st.file_uploader("Upload Resume PDF", type=["pdf"])

job_description = st.text_area("Paste Job Description")

# ==================================================
# MAIN LOGIC
# ==================================================

if uploaded_file and job_description:

    # Extract Resume Text
    resume_text = extract_text_from_pdf(uploaded_file)

    # Clean
    resume_clean = clean_text(resume_text)

    job_clean = clean_text(job_description)

    # Skills
    job_skills = extract_skills(job_clean)

    resume_skills = extract_skills(resume_clean)

    matched = list(set(job_skills) & set(resume_skills))

    missing = list(set(job_skills) - set(resume_skills))

    # Scores
    semantic_score = compute_similarity(resume_clean, job_clean)

    skill_score = (len(matched) / len(job_skills)) * 100

    # Verdict
    if semantic_score > 75:
        verdict = "Strong Match ✅"

    elif semantic_score > 50:
        verdict = "Moderate Match ⚠️"

    else:
        verdict = "Low Match ❌"

    # ==================================================
    # DISPLAY RESULTS
    # ==================================================

    st.subheader("📊 Analysis Results")

    st.write(f"### Semantic Match Score: {semantic_score}%")

    st.write(f"### Skill Match Score: {round(skill_score,2)}%")

    st.write(f"### Verdict: {verdict}")

    st.write("## ✅ Matched Skills")

    st.write(matched)

    st.write("## ❌ Missing Skills")

    st.write(missing)

    # ==================================================
    # CHART
    # ==================================================

    fig, ax = plt.subplots()

    labels = ['Matched', 'Missing']

    counts = [len(matched), len(missing)]

    ax.bar(labels, counts)

    st.pyplot(fig)

Writing app.py


In [ ]:
# !streamlit run app.py & npx localtunnel --port 8501

from pyngrok import ngrok

# 1. Plug in your token here
ngrok.set_auth_token("3DQMvg7DN2UTMoMvgHFxvhcT66M_7v4cXJhmhSmNNaXvK447J")

# 2. Open the tunnel
public_url = ngrok.connect(8501)
print(f"Click here to open your app: {public_url}")

# 3. Run Streamlit
!streamlit run app.py

Click here to open your app: NgrokTunnel: "https://handbook-marry-hazy.ngrok-free.dev" -> "http://localhost:8501"


2026-05-08 03:16:25.617 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.245.48.96:8501

Loading weights: 100% 103/103 [00:00<00:00, 1771.88it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100% 103/103 [00:00<00:00, 1392.71it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+